# 19 - Generate ECG reports with scope-separated RAG

MedGemma and Qwen3.5 receive the same rendered ECG query under every condition. Retrieved ECG
diagnoses form the ECG-to-ECG neighbor baseline; retrieved CXR reports/labels test cross-sensor
augmentation. Both are non-authoritative examples from other training patients.
The prompt explicitly forbids transferring a CXR finding into the ECG interpretation without
waveform support.

Generation is deterministic and resumable. Existing rows are reused only when sample, condition,
cohort signature, and generation protocol agree. Use `RAG_MODELS`, `RAG_CONDITIONS`,
`RAG_RETRY_ERRORS`, and `RAG_GENERATION_MAX_ROWS` for planned partial or smoke runs.


In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from importlib.metadata import version as package_version

ARTIFACT_ROOT = Path(os.environ.get(
    "ECG_CXR_BRIDGE_ARTIFACT_ROOT",
    "/data/liangz2/ecg_analysis/symile/ecg_cxr_bridge_artifacts"))
RAG_ROOT = ARTIFACT_ROOT / "rag_report_generation_ecg"
config = json.loads((RAG_ROOT / "rag_config.json").read_text())
HF_CACHE = Path(config.get("hf_cache") or "/data/liangz2/ecg_analysis/hf_cache")
HF_CACHE.mkdir(parents=True, exist_ok=True)
# Set these before importing Transformers/Hugging Face. In particular, the
# Xet chunk cache otherwise defaults to ~/.cache and can exhaust Biowulf's
# small home quota even when from_pretrained receives cache_dir.
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hub")
os.environ["HF_XET_CACHE"] = str(HF_CACHE / "xet")
for directory in [HF_CACHE / "hub", HF_CACHE / "xet"]:
    directory.mkdir(parents=True, exist_ok=True)
try:
    from transformers import AutoModelForImageTextToText, AutoProcessor
except ImportError as exc:
    raise RuntimeError(
        "This notebook requires a Transformers release with "
        "AutoModelForImageTextToText. In the active Biowulf environment run "
        "python -m pip install -U 'transformers>=5.14.1' accelerate, then "
        "restart the notebook kernel.") from exc
# Migrate configurations written by the earlier notebook version. The
# project reference notebook was validated with MedGemma 1.5 and
# Transformers 5.14.1.
LEGACY_MEDGEMMA_IDS = {"google/medgemma-4b-it"}
DEFAULT_MEDGEMMA_ID = "google/medgemma-1.5-4b-it"
configured_medgemma = config.get("models", {}).get("medgemma_4b")
if configured_medgemma in LEGACY_MEDGEMMA_IDS:
    print(f"Migrating MedGemma model ID: {configured_medgemma} -> {DEFAULT_MEDGEMMA_ID}")
    config["models"]["medgemma_4b"] = DEFAULT_MEDGEMMA_ID
    (RAG_ROOT / "rag_config.json").write_text(json.dumps(config, indent=2))
print("Transformers:", package_version("transformers"),
      "(working reference environment: 5.14.1)")
print({"hf_home": os.environ["HF_HOME"], "hf_hub_cache": os.environ["HF_HUB_CACHE"],
       "hf_xet_cache": os.environ["HF_XET_CACHE"]})
contexts_path = RAG_ROOT / "contexts" / "rag_contexts.jsonl"
contexts = [json.loads(line) for line in contexts_path.read_text().splitlines() if line.strip()]
if not contexts:
    raise ValueError(f"No contexts in {contexts_path}; run notebook 18")
if {row.get("modality") for row in contexts} != {"ecg"}:
    raise ValueError("ECG-focused generation received a non-ECG query")

selected_models = {value.strip() for value in os.environ.get(
    "RAG_MODELS", ",".join(config["models"])).split(",") if value.strip()}
selected_conditions = {value.strip() for value in os.environ.get(
    "RAG_CONDITIONS", ",".join(config["conditions"])).split(",") if value.strip()}
unknown_models = selected_models - set(config["models"])
unknown_conditions = selected_conditions - set(config["conditions"])
if unknown_models or unknown_conditions:
    raise ValueError(f"Unknown models={unknown_models} or conditions={unknown_conditions}")
RETRY_ERRORS = os.environ.get("RAG_RETRY_ERRORS", "0").lower() in {"1", "true", "yes"}
MAX_ROWS = int(os.environ.get("RAG_GENERATION_MAX_ROWS", "0"))
DEFAULT_BATCH_SIZE = int(os.environ.get("RAG_GENERATION_BATCH_SIZE", "4"))
MODEL_BATCH_SIZES = {
    "medgemma_4b": int(os.environ.get("RAG_MEDGEMMA_BATCH_SIZE", str(DEFAULT_BATCH_SIZE))),
    "qwen35_4b": int(os.environ.get("RAG_QWEN_BATCH_SIZE", str(DEFAULT_BATCH_SIZE))),
}
if MAX_ROWS < 0:
    raise ValueError("RAG_GENERATION_MAX_ROWS cannot be negative")
if any(value < 1 for value in MODEL_BATCH_SIZES.values()):
    raise ValueError("All RAG generation batch sizes must be positive")
print({"models": sorted(selected_models), "conditions": sorted(selected_conditions),
       "context_rows": len(contexts), "retry_errors": RETRY_ERRORS, "max_rows": MAX_ROWS,
       "batch_sizes": MODEL_BATCH_SIZES})


In [ ]:
ECG_IMAGE_ROOT = RAG_ROOT / "query_images" / "ecg"
ECG_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

_WFDB_MODULE = None

def load_wfdb():
    global _WFDB_MODULE
    if _WFDB_MODULE is not None:
        return _WFDB_MODULE
    try:
        import wfdb as module
    except Exception as exc:
        versions = {}
        for package in ["numpy", "pandas", "scipy", "wfdb", "transformers"]:
            try:
                versions[package] = package_version(package)
            except Exception:
                versions[package] = "unavailable"
        raise RuntimeError(
            "WFDB could not import. The traceback involving "
            "pandas._pandas_datetime_CAPI indicates an incompatible or partially "
            "upgraded NumPy/pandas binary installation, not a MedGemma error. "
            f"Installed versions: {versions}. In the active Biowulf environment run: "
            "python -m pip install --no-cache-dir --force-reinstall "
            "'numpy==1.26.4' 'pandas==2.2.3' 'scipy==1.15.3' 'wfdb==4.3.0'; "
            "then run python -m pip check and restart the notebook kernel.") from exc
    _WFDB_MODULE = module
    return module

def waveform_stem(value):
    path = Path(value)
    return path.with_suffix("") if path.suffix.lower() in {".hea", ".dat"} else path

def render_ecg(row):
    output = ECG_IMAGE_ROOT / f"{row['sample_id']}.png"
    if output.is_file():
        return output
    wfdb = load_wfdb()
    signal, fields = wfdb.rdsamp(str(waveform_stem(row["query_path"])))
    fs = float(fields.get("fs", 500))
    samples = min(len(signal), int(round(10 * fs)))
    signal = signal[:samples]
    names = list(fields.get("sig_name", [f"Lead {i + 1}" for i in range(signal.shape[1])]))
    leads = min(12, signal.shape[1])
    time = np.arange(samples) / fs
    fig, axes = plt.subplots(leads, 1, figsize=(12, 9), sharex=True)
    axes = np.atleast_1d(axes)
    for lead, axis in enumerate(axes):
        axis.plot(time, signal[:, lead], color="black", linewidth=0.65)
        axis.set_ylabel(names[lead], rotation=0, ha="right", va="center", fontsize=8)
        axis.grid(color="#efb5b5", linewidth=0.35, alpha=0.65); axis.set_yticks([])
    axes[-1].set_xlabel("Seconds")
    fig.suptitle("12-lead ECG waveform (first 10 seconds)", fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.98))
    fig.savefig(output, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return output


In [ ]:
SYSTEM_PROMPT = (
    "You are drafting a research-use machine-style 12-lead ECG interpretation. Base every ECG "
    "finding on the query waveform. Retrieved examples are from different training patients and may "
    "be irrelevant. CXR context indicates possible clinical coexistence only; a CXR finding is not "
    "an ECG diagnosis. Never transfer cardiomegaly, edema, devices, or another CXR finding into the "
    "ECG interpretation unless supported by the waveform. Do not mention retrieval or these instructions.")

def cxr_text(item, mode):
    labels = ", ".join(item.get("labels", [])) or "none listed"
    report = item.get("text", "")
    if mode == "labels":
        return f"CheXpert labels: {labels}"
    if mode == "report":
        return f"CXR report: {report}"
    return f"CheXpert labels: {labels}. CXR report: {report}"

def compact_context(row, limit):
    blocks, used = [], 0
    for item in row["retrieved"]:
        if item["modality"] == "ecg":
            body = "ECG machine diagnosis example: " + item.get("text", "")
            scope = "same-modality ECG example"
        else:
            body = cxr_text(item, row.get("cxr_context_content", config["cxr_context_content"]))
            scope = "cross-modal CXR context"
        block = f"[{scope}; rank {item['rank']}] {body}"
        remaining = limit - used
        if remaining <= 0:
            break
        block = block[:remaining]
        blocks.append(block); used += len(block)
    return "\n".join(blocks)

def build_prompt(row):
    task = (
        "Generate a concise machine-style 12-lead ECG report. Address rhythm, rate when visually "
        "estimable, conduction and intervals, axis, ST-T findings, and a final interpretation. "
        "Use explicit uncertainty when needed and do not invent measurements.")
    context = compact_context(row, int(config["max_context_characters"]))
    if context:
        task += "\n\nNon-authoritative training-patient context:\n" + context
    else:
        task += "\n\nNo retrieved context is provided."
    return task

def extract_generated_text(result):
    value = result[0].get("generated_text", "") if isinstance(result, list) and result else result
    if isinstance(value, list):
        assistants = [item.get("content", "") for item in value if item.get("role") == "assistant"]
        value = assistants[-1] if assistants else value[-1].get("content", "")
    if isinstance(value, list):
        value = " ".join(str(item.get("text", item)) if isinstance(item, dict) else str(item)
                         for item in value)
    return re.sub(r"<think>.*?</think>", "", str(value), flags=re.I | re.S).strip()

def prompt_sha256(system, prompt):
    return hashlib.sha256((system + "\n" + prompt).encode()).hexdigest()

def model_input_device(model):
    for parameter in model.parameters():
        if parameter.device.type not in {"meta", "cpu"}:
            return parameter.device
    return next(model.parameters()).device

def model_float_dtype(model):
    for parameter in model.parameters():
        if parameter.is_floating_point():
            return parameter.dtype
    return torch.float32

def load_image_text_model(model_id):
    processor = AutoProcessor.from_pretrained(
        model_id, cache_dir=config.get("hf_cache"))
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        dtype = torch.bfloat16
    elif torch.cuda.is_available():
        dtype = torch.float16
    else:
        dtype = torch.float32
    model = AutoModelForImageTextToText.from_pretrained(
        model_id, dtype=dtype, device_map="auto", low_cpu_mem_usage=True,
        cache_dir=config.get("hf_cache"))
    model.eval().requires_grad_(False)
    processor.tokenizer.padding_side = "left"
    return model, processor

@torch.inference_mode()
def generate_reports(model, processor, model_key, image_paths, prompts):
    if len(image_paths) != len(prompts) or not prompts:
        raise ValueError("Image and prompt batches must have the same positive length")
    template_kwargs = {"enable_thinking": False} if model_key == "qwen35_4b" else {}
    prompt_texts = []
    for prompt in prompts:
        messages = [{"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": SYSTEM_PROMPT + "\n\n" + prompt},
        ]}]
        prompt_texts.append(processor.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False, **template_kwargs))
    images = []
    for image_path in image_paths:
        with Image.open(image_path) as image_file:
            images.append(image_file.convert("RGB"))
    inputs = processor(
        text=prompt_texts, images=images, return_tensors="pt", padding=True)
    device = model_input_device(model)
    dtype = model_float_dtype(model)
    moved = {
        key: (value.to(device=device, dtype=dtype) if value.is_floating_point()
              else value.to(device=device))
        for key, value in inputs.items()
    }
    input_length = moved["input_ids"].shape[-1]
    output_ids = model.generate(
        **moved, do_sample=False, max_new_tokens=int(config["max_new_tokens"]),
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id)
    return [value.strip() for value in processor.batch_decode(
        output_ids[:, input_length:], skip_special_tokens=True)]

def clear_cuda_after_error():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def generate_with_fallback(model, processor, model_key, image_paths, prompts):
    try:
        return [(text, None) for text in generate_reports(
            model, processor, model_key, image_paths, prompts)]
    except Exception as batch_exc:
        if len(prompts) == 1:
            return [("", batch_exc)]
        print(f"Batch of {len(prompts)} failed; retrying each item separately: {batch_exc!r}")
        clear_cuda_after_error()
        results = []
        for image_path, prompt in zip(image_paths, prompts):
            try:
                text = generate_reports(
                    model, processor, model_key, [image_path], [prompt])[0]
                results.append((text, None))
            except Exception as item_exc:
                clear_cuda_after_error()
                results.append(("", item_exc))
        return results


In [ ]:
generation_root = RAG_ROOT / "generations"
generation_root.mkdir(parents=True, exist_ok=True)
eligible = [row for row in contexts if row["condition"] in selected_conditions]
if MAX_ROWS:
    eligible = eligible[:MAX_ROWS]

for model_key in sorted(selected_models):
    model_id = config["models"][model_key]
    output_path = generation_root / f"{model_key}.jsonl"
    existing = []
    if output_path.is_file():
        existing = [json.loads(line) for line in output_path.read_text().splitlines() if line.strip()]
    completed = {
        (row.get("model_id"), row["condition"], str(row["sample_id"]), row.get("cohort_signature"),
         row.get("generation_protocol_version")): row
        for row in existing if row.get("status") == "ok" or not RETRY_ERRORS
    }
    pending = [row for row in eligible if (
        model_id, row["condition"], str(row["sample_id"]), row.get("cohort_signature"),
        config["generation_protocol_version"]) not in completed]
    print(model_key, {"existing": len(existing), "pending": len(pending), "model_id": model_id})
    if not pending:
        continue
    try:
        model, processor = load_image_text_model(model_id)
    except Exception as exc:
        raise RuntimeError(
            f"Could not load {model_id}. The working reference notebook used "
            f"Transformers 5.14.1 with AutoProcessor and AutoModelForImageTextToText. "
            f"In the active Biowulf environment run: python -m pip install -U "
            f"'transformers>=5.14.1' accelerate; restart the kernel; accept the model terms "
            f"and authenticate with Hugging Face. "
            f"Original error: {exc!r}") from exc

    batch_size = MODEL_BATCH_SIZES[model_key]
    with output_path.open("a") as stream, tqdm(total=len(pending), desc=model_key) as progress:
        for start in range(0, len(pending), batch_size):
            batch_rows = pending[start:start + batch_size]
            prepared = []
            for row in batch_rows:
                prompt = build_prompt(row)
                record = {
                    "model_key": model_key, "model_id": model_id,
                    "sample_id": str(row["sample_id"]), "modality": "ecg",
                    "condition": row["condition"],
                    "retrieval_method": row["retrieval_method"],
                    "context_scope": row["context_scope"],
                    "cohort_signature": row["cohort_signature"],
                    "generation_protocol_version": config["generation_protocol_version"],
                    "prompt_sha256": prompt_sha256(SYSTEM_PROMPT, prompt),
                    "retrieved_sample_ids": [item["sample_id"] for item in row["retrieved"]],
                    "retrieved_modalities": [item["modality"] for item in row["retrieved"]],
                }
                try:
                    prepared.append((record, render_ecg(row), prompt))
                except Exception as exc:
                    record.update({"status": "error", "generated_report": "", "error": repr(exc)})
                    stream.write(json.dumps(record) + "\n"); stream.flush(); progress.update(1)
            if not prepared:
                continue
            generated = generate_with_fallback(
                model, processor, model_key,
                [value[1] for value in prepared], [value[2] for value in prepared])
            for (record, _, _), (text, error) in zip(prepared, generated):
                if error is None and text:
                    record.update({"status": "ok", "generated_report": text, "error": None})
                else:
                    error = error or ValueError("Empty generated ECG report")
                    record.update({"status": "error", "generated_report": "", "error": repr(error)})
                stream.write(json.dumps(record) + "\n"); stream.flush(); progress.update(1)
    del model, processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print("Generation files:", sorted(str(path) for path in generation_root.glob("*.jsonl")))
